# 單元 9-3 二維陣列初始化與參照共用致命陷阱

- **適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
- **對應教材**：第 13 章 二維陣列初始化（第 47 頁）
- **核心目標**：
  1. 徹底剖析 `[[0] * C] * R` 產生的「牽一髮動全身」修改一格整直行暴走的世紀大慘劇。
  2. 搞懂乘法複製 `* R` 背後的指針別名機制（只複製了同一把抽屜鑰匙，而非創造獨立置物盒）。
  3. 熟練掌握官方唯一推薦正解：二維列表生成式 `[[0] * C for _ in range(R)]`。
  4. 使用科學的 `id()` 門牌驗證法，親眼見證不同橫列在物理記憶體中的完全獨立性。
  5. 提早建立二維深淺拷貝預防針，學會用 `[row[:] for row in grid]` 實現安全的二維陣列深層備份。

---
### 🧭 單元導航地圖
1. **9.3.1 初學者的噩夢：`[[0] * C] * R` 牽一髮動全身的慘劇**
2. **9.3.2 官方標準解法：列表生成式初始化 `[[0] * C for _ in range(R)]`**
3. **9.3.3 二維串列的身分驗證：`id(grid[0])` 與 `id(grid[1])`**
4. **9.3.4 特殊初始值網格建構：預設字符、棋盤格與漸增流水號**
5. **9.3.5 二維陣列的深淺拷貝危機與防禦實踐**

### 9.3.1 初學者的噩夢：`[[0] * C] * R` 牽一髮動全身的慘劇

當我們需要建立一個 $3 \times 3$ 且內容全部都是 `0` 的空白網格時，很多同學會想起第七章和第八章學過的「串列乘法」：
- `[0] * 3` 可以產生 `[0, 0, 0]`，那我在外面再乘一個 3，不就是 3 排了嗎？
```python
bad_grid = [[0] * 3] * 3  # 💀 千萬不要這樣寫！這是定時炸彈！
```

#### 靈異現象重現：
當你把這張網格印出來時，長相非常完美：`[[0, 0, 0], [0, 0, 0], [0, 0, 0]]`。
但是，當你試圖在第 0 列第 0 行下子：
```python
bad_grid[0][0] = 9
```
再把網格印出來，你會看到令人毛骨悚然的一幕：
```text
[9, 0, 0]
[9, 0, 0]  <-- 為什麼你也變 9 了？！
[9, 0, 0]  <-- 你怎麼也跟著變 9 了？！
```

#### 底層慘劇解析：
在第八章 8.6 我們學過「參照與別名」。外層的 `* 3`，並沒有製造出 3 個新的內層抽屜，而是把**同一個小串列的鑰匙複製了 3 份**！
換句話說，`bad_grid[0]`、`bad_grid[1]`、`bad_grid[2]` 根本是同一個小抽屜的 3 個別名！修改了任何一列的第 0 格，其他所有列的第 0 格會同步被竄改！這在 APCS 考試中如果沒發現，整題會直接拿到 0 分！

In [ ]:
# [2] Code 範例：慘劇重現——[[0]*C]*R 的連動副作用

# 宣告看似正常的 3x3 矩陣
trap_grid = [[0] * 3] * 3

print("初始化後的樣子：", trap_grid)

# 我們只打算修改第一列第一格 (0, 0)
trap_grid[0][0] = 777

print("\n=== 修改 trap_grid[0][0] = 777 之後 ===")
for row in trap_grid:
    print(row)  # 驚悚！每一列的第一個數字都變成了 777！

In [ ]:
# [3] Code 填空題
# 任務：驗證 trap_grid 的第 0 列與第 1 列是不是同一個物件

trap_grid = [[0] * 2] * 2

# 使用 is 運算子判斷第 0 列與第 1 列的身分
is_same_object = trap_grid[0] ___ trap_grid[1]

print("第 0 列與第 1 列是同一個抽屜嗎？", is_same_object)  # 應輸出 True

In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 執行有陷阱的宣告 bad_grid = [[1, 2]] * 3。
# 請修改 bad_grid[1][1] = 99。
# 接著輸出 bad_grid[0] 的內容，印證它是否也受到了污染。
#
# 【公開測試資料 1】
# 設定：依照題意宣告並修改
# 預期輸出：[1, 99]
#
# 【公開測試資料 2】
# 設定：若初值為 bad_grid = [[0, 0, 0]] * 2，修改 bad_grid[0][2] = 5
# 預期輸出 bad_grid[1] 的內容：
# [0, 0, 5]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
bad_grid = [[1, 2]] * 3
bad_grid[1][1] = 99
print(bad_grid[0])


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 請撰寫一段程式碼，展示利用乘法建立二維網格的嚴重錯誤：
# 建立一個 4x4 的全 0 網格 grid = [[0]*4]*4。
# 當你試圖只在對角線上的位置 grid[2][2] 填入 1 時，
# 印出整張網格，並說明為什麼第 2 直行（column 2）會整條變成 1。
# （本題為自由挑戰題，無公開測資，請自行撰寫程式碼與註解）
# ==========================================
# 請在下方撰寫你的程式碼：


### 9.3.2 官方標準解法：列表生成式初始化 `[[0] * C for _ in range(R)]`

既然 `* R` 會造成鑰匙共用的大災難，那我們到底該如何正確初始化一個 $R$ 列 $C$ 行的全 0 網格呢？

Python 官方與所有資深選手唯一指定的**標準正解**是：
```python
safe_grid = [[0] * C for _ in range(R)]
```

#### 為什麼列表生成式不會翻車？
- 注意看順序：外層是 `for _ in range(R)`！
- 在這個迴圈中，每跑一回合，Python 就會**「重新執行一次 `[0] * C`」**！
- 每次重新執行，Python 就會在記憶體中開闢一個**全新的內層串列（全新的實體小抽屜）**。
- 總共跑了 $R$ 回合，就紮紮實實造出了 $R$ 個獨立的小抽屜，每個小抽屜的記憶體門牌號碼（id）都完全不同！

#### 成果驗證：
現在你可以放心大膽地修改 `safe_grid[0][0] = 9`，其他橫列依然安穩地保持著 `0`，這才是真正的二維陣列獨立空間！

In [ ]:
# [2] Code 範例：正確使用二維列表生成式初始化

R, C = 3, 4
# 正確寫法：每回合全新建立一個 [0] * C
safe_grid = [[0] * C for _ in range(R)]

print("初始化後的安全網格：")
for row in safe_grid:
    print(row)

# 測試修改單一格子
safe_grid[0][0] = 99
safe_grid[1][2] = 88

print("\n=== 修改 (0, 0) 與 (1, 2) 後 ===")
for row in safe_grid:
    print(row)  # 完美！只有被指定的格子改變，其他列毫無波及！

In [ ]:
# [3] Code 填空題
# 任務：建立一個 4 列 3 行、初始值全部為 -1 的安全二維陣列 memo

R, C = 4, 3
# 請填入正確的列表生成式語法
memo = [[-1] * ___ for _ in range(___)]

# 修改第 2 列第 1 行為 100
memo[2][1] = 100

print("第 0 列：", memo[0])  # 應依然是 [-1, -1, -1]
print("第 2 列：", memo[2])  # 應為 [-1, 100, -1]

In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 給定列數 R = 3, 行數 C = 3。
# 請使用列表生成式建立一個初始值皆為 0 的安全矩陣 identity。
# 接著使用一個 for 迴圈，將對角線上的三格 (i, i) 全部修改為 1。
# 最後逐列印出 identity。
#
# 【公開測試資料 1】
# 設定：R = 3, C = 3
# 預期輸出：
# [1, 0, 0]
# [0, 1, 0]
# [0, 0, 1]
#
# 【公開測試資料 2】
# 設定：若 R = 2, C = 2
# 預期輸出：
# [1, 0]
# [0, 1]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
R, C = 3, 3
identity = [[0] * C for _ in range(R)]
for i in range(R):
    identity[i][i] = 1

for row in identity:
    print(row)


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 在圖論與迷宮尋路演算法中，常需要一個「拜訪標記陣列（Visited Matrix）」：
# 請使用二維列表生成式，建立一個 3 列 5 行且初始值全部為 False 的布林網格 visited。
# 請將邊界上的四個角落點全部標記為 True。
# 最後印出整張 visited 網格。
# （本題為自由挑戰題，無公開測資，請自行撰寫程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：


### 9.3.3 二維串列的身分驗證：`id(grid[0])` 與 `id(grid[1])`

如果有人不信邪，覺得「寫 `* R` 和寫 `for _ in range(R)` 明明印出來都一樣是 0，為什麼非要用生成式不可？」

這時只要拿出科學利器：**`id()` 記憶體身分證檢驗**！

#### 顯微鏡下的真實世界：
1. **在錯誤寫法 `bad = [[0] * 3] * 3` 中**：
   - `id(bad[0])` = `140239481234`
   - `id(bad[1])` = `140239481234`（完全一模一樣的門牌號碼！）
   - `bad[0] is bad[1]` 結果是無庸置疑的 **`True`**！這直接宣告了它們就是同一個串列。

2. **在正確生成式寫法 `good = [[0] * 3 for _ in range(3)]` 中**：
   - `id(good[0])` = `140239481234`
   - `id(good[1])` = `140239485678`（截然不同的門牌號碼！）
   - `good[0] is good[1]` 結果是安心的 **`False`**！這證明了每一橫列都是享有獨立記憶體空間的健康個體。

這就是程式底層原理的魅力所在——唯有知其然，更知其所以然，才能在考場與開發中百毒不侵！

In [ ]:
# [2] Code 範例：記憶體門牌號碼比對實驗

# 1. 錯誤寫法實驗
bad = [[0] * 3] * 3
print("=== 錯誤寫法的 id 檢驗 ===")
print("id(bad[0]) =", id(bad[0]))
print("id(bad[1]) =", id(bad[1]))
print("bad[0] is bad[1] ?", bad[0] is bad[1])  # True

# 2. 正確寫法實驗
good = [[0] * 3 for _ in range(3)]
print("\n=== 正確寫法的 id 檢驗 ===")
print("id(good[0]) =", id(good[0]))
print("id(good[1]) =", id(good[1]))
print("good[0] is good[1] ?", good[0] is good[1])  # False

In [ ]:
# [3] Code 填空題
# 任務：檢查給定的網格 test_grid，是否各列擁有獨立的記憶體空間

test_grid = [[0] * 2 for _ in range(2)]

# 檢查第 0 列與第 1 列的 id 是否不相等
is_independent = id(test_grid[0]) ___ id(test_grid[1])

print("各列是否擁有獨立記憶體？", is_independent)  # 應為 True

In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 宣告 gridA = [[0]*2]*2 與 gridB = [[0]*2 for _ in range(2)]。
# 請分別印出：
# 1. gridA[0] is gridA[1] 的布林值結果
# 2. gridB[0] is gridB[1] 的布林值結果
#
# 【公開測試資料 1】
# 設定：依題意建立 gridA 與 gridB
# 預期輸出：
# True
# False
#
# 【公開測試資料 2】
# 設定：若為 3x3 網格的比對結果
# 預期輸出：
# True
# False
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
gridA = [[0] * 2] * 2
gridB = [[0] * 2 for _ in range(2)]

print(gridA[0] is gridA[1])
print(gridB[0] is gridB[1])


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 請撰寫一段程式碼，設計一個檢驗機制：
# 給定一個長寬為 4x4 的未知來源網格 matrix。
# 請利用迴圈逐一比對相鄰兩列的 id（如列 0 與列 1、列 1 與列 2...），
# 若發現任何兩列的 id 相同，立刻印出「警報：發現危險的參照共用！」並中斷迴圈；
# 若全部相鄰列的 id 都不相同，則印出「安全：各列皆為獨立副本」。
# （本題為自由挑戰題，無公開測資，請自行構思變數並撰寫檢驗程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：


### 9.3.4 特殊初始值網格建構：預設字符、棋盤格與漸增流水號

在掌握了列表生成式初始化之後，我們就不再受限於只能建立「全 0 網格」了！
在各種解題情境中，往往需要多樣化的初始網格：

#### 實用特殊初始網格範例：
1. **全字元網格（例如字元迷宮的空地 `'.'` 或圍牆 `'#'`）**：
   ```python
   maze = [['.'] * C for _ in range(R)]
   ```
2. **連續流水號矩陣（例如 $3 \times 3$ 填入 1 到 9）**：
   利用公式 `流水號 = r * C + c + 1`，可以在生成式中一行造出循序編號的九宮格：
   ```python
   ordered_grid = [[r * C + c + 1 for c in range(C)] for r in range(R)]
   ```
3. **黑白交錯棋盤格（西洋棋盤圖案）**：
   利用座標相加的奇偶性 `(r + c) % 2`：偶數放 `'B'`、奇數放 `'W'`。

只要善用二維列表生成式，任何複雜規律的幾何初始地圖都能在彈指之間完成！

In [ ]:
# [2] Code 範例：生成特殊初始值網格

R, C = 3, 3

# 1. 連續流水號九宮格：1 到 9
num_grid = [[r * C + c + 1 for c in range(C)] for r in range(R)]
print("=== 連續流水號九宮格 ===")
for row in num_grid:
    print(*row)

# 2. 西洋棋黑白交錯棋盤格
chess_board = [["B" if (r + c) % 2 == 0 else "W" for c in range(4)] for r in range(4)]
print("\n=== 4x4 西洋棋盤 ===")
for row in chess_board:
    print(*row)

In [ ]:
# [3] Code 填空題
# 任務：建立一個 2 列 4 行全部由字元 "#" 組成的圍牆網格 wall

R, C = 2, 4
# 請填入初始字元與生成式
wall = [[___] * C for _ in range(___)]

for r in wall:
    print(*r)

In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 請使用二維列表生成式，建立一個 2 列 3 行的連續流水號矩陣
# 內容為：
# 1 2 3
# 4 5 6
# 並逐列解包印出。
#
# 【公開測試資料 1】
# 設定：R = 2, C = 3
# 預期輸出：
# 1 2 3
# 4 5 6
#
# 【公開測試資料 2】
# 設定：若為 R = 1, C = 4
# 預期輸出：
# 1 2 3 4
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
R, C = 2, 3
grid = [[r * C + c + 1 for c in range(C)] for r in range(R)]
for row in grid:
    print(*row)


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 請使用列表生成式建立一個 5x5 的矩陣：
# 規則：若座標位於外框邊界（即 r == 0 或 r == 4 或 c == 0 或 c == 4）填入 1；
# 內部格子則填入 0。
# 逐列印出這個加框矩陣。
# （本題為自由挑戰題，無公開測資，請自行構思三元運算式並完成撰寫）
# ==========================================
# 請在下方撰寫你的程式碼：


### 9.3.5 二維陣列的深淺拷貝危機與防禦實踐

在第八章 8.6 中，我們曾學過一個備份利器：`backup = original.copy()` 或 `backup = original[:]`。
在那時我們強調過：**這對一維純量串列來說是 100% 安全獨立的**。

#### 但在二維陣列中，這個方法會再次翻車！
為什麼呢？因為 `.copy()` 與 `[:]` 都是**淺拷貝（Shallow Copy）**！
- 想像外層串列是一個大資料夾，裡面裝著多個小筆記本（內層串列）。
- 當你執行 `grid_copy = grid.copy()` 時，影印機確實影印了一個「全新編號的大資料夾」。
- **但是！大資料夾裡面放的，依然是原本那幾本小筆記本的鑰匙！**
- 結果：修改 `grid_copy[0][0] = 999`，正本的 `grid[0][0]` 竟然也跟著被改掉了！

#### 二維專屬的安全複製法（Deep Copy 生成式版）：
在不引用外部模組的情況下，Python 最乾淨又標準的二維深層備份寫法是：
```python
safe_backup = [row[:] for row in grid]
```
- 利用外層列表生成式，走訪原本的每一個橫列 `row`。
- 在每一回合，對該橫列單獨做一次切片拷貝 `row[:]`，生成一個全新獨立的小抽屜！
這樣做出來的二維副本，外層是新的、內層每一列也全都是新的，達到徹底的雙重隔離防護！

In [ ]:
# [2] Code 範例：二維陣列使用普通 copy() 翻車 vs 安全備份

origin = [[1, 2], [3, 4]]

# 1. 錯誤的淺拷貝示範
shallow_backup = origin.copy()
shallow_backup[0][0] = 888
print("淺拷貝修改後，origin[0][0] 也被改了！：", origin[0][0])  # 888！

# 2. 正確的二維生成式深層備份
origin_fixed = [[1, 2], [3, 4]]
deep_backup = [row[:] for row in origin_fixed]

deep_backup[0][0] = 999
print("\n二維深層備份修改後：")
print("副本修改為：", deep_backup[0])  # [999, 2]
print("正本依然是：", origin_fixed[0]) # [1, 2] 完好無損！

In [ ]:
# [3] Code 填空題
# 任務：使用列表生成式搭配切片拷貝，建立二維矩陣 matrix 的真正獨立副本

matrix = [[10, 20], [30, 40]]

# 請填入對每列進行切片拷貝的語法
backup_matrix = [row[___] for row in ___]

backup_matrix[0][1] = 0

print("正本第一列：", matrix[0])        # 應為 [10, 20]
print("副本第一列：", backup_matrix[0]) # 應為 [10, 0]

In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 給定二維原始資料 raw_grid = [[5, 6], [7, 8]]。
# 請使用 [r.copy() for r in raw_grid] 建立安全副本 safe_copy。
# 將 safe_copy 內部所有大於等於 7 的數值替換為 0。
# 最後印出 raw_grid 與 safe_copy。
#
# 【公開測試資料 1】
# 設定：raw_grid = [[5, 6], [7, 8]]
# 預期輸出：
# 原本: [[5, 6], [7, 8]]
# 副本: [[5, 6], [0, 0]]
#
# 【公開測試資料 2】
# 設定：raw_grid = [[7, 7]]
# 預期輸出：
# 原本: [[7, 7]]
# 副本: [[0, 0]]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
raw_grid = [[5, 6], [7, 8]]
safe_copy = [r.copy() for r in raw_grid]

for r in range(len(safe_copy)):
    for c in range(len(safe_copy[0])):
        if safe_copy[r][c] >= 7:
            safe_copy[r][c] = 0

print(f"原本: {raw_grid}")
print(f"副本: {safe_copy}")


In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 模擬 APCS 解題時的狀態回溯機制：
# 棋局當前狀態 state = [["X", "O"], [".", "X"]]。
# 請建立一份安全的獨立深層副本 trial_state。
# 在 trial_state[1][0] 試著落子 "O"。
# 驗證 state[1][0] 依然保持為 "."，並印出兩張棋盤的狀態對照。
# （本題為自由挑戰題，無公開測資，請自行完成程式碼撰寫）
# ==========================================
# 請在下方撰寫你的程式碼：


## 🎯 學習總結與通關簽到

恭喜你徹底攻克了二維陣列最著名的暗礁大陷阱，成功通關 **單元 9-3 二維陣列初始化與參照共用致命陷阱**！

### 核心避坑防禦守則：
1. **致命禁忌**：絕對不要寫 `[[0] * C] * R`！這會導致所有橫列共用同一個內層抽屜。
2. **官方標準**：永遠使用列表生成式 `[[0] * C for _ in range(R)]`，保證每列獨立。
3. **身分檢驗**：利用 `id(grid[0]) != id(grid[1])` 科學驗證各橫列的物理獨立性。
4. **特殊網格**：掌握流水號公式 `r * C + c + 1` 與奇偶相間黑白西洋棋盤生成技巧。
5. **二維安全拷貝**：二維備份必須使用 `[row[:] for row in grid]`，徹底杜絕淺拷貝共用污染！

---榮譽授予【二維記憶體拆彈專家徽章】💣！下一單元我們將進入二維演算法的核心走訪技術：**9-4 二維網格雙重走訪與行列統計**！